# Distillation : un élève rapide qui imite le zero-shot de la thèse

**Objectif.** Le zero-shot `MoritzLaurer/deberta-v3-large-zeroshot-v2.0` (l'**enseignant**) lit chaque verbatim **28 fois**, une fois par émotion. C'est ce qui coûte 1 à 2 minutes par verbatim sur CPU. Ce notebook entraîne un **élève** qui lit le verbatim **une seule fois** et sort directement :

1. les **28 émotions**, chacune évaluée indépendamment (comme le zero-shot en multi-label) ;
2. les **5 scores factoriels de la thèse**, calculés à partir de ces 28 émotions par une couche mathématique **figée**, qui reproduit exactement le calcul de la thèse.

```
ENTRAÎNEMENT (GPU, une fois)                      PRODUCTION (CPU, chaque nouveau verbatim)
160 000 verbatims ─► élève ─► 28 émotions         verbatim ─► élève (1 lecture) ─► 28 émotions
                        │                                                         │
    comparées aux 28 émotions du zero-shot                           couche figée de la thèse
                        │                                       (rangs ─► poids de Bartlett)
              correction de l'élève                                               │
                                                                        5 scores factoriels
```

**Ce que l'élève peut et ne peut pas faire.** Il apprend à **reproduire** le zero-shot, avec ses qualités et ses défauts. Toutes les métriques de ce notebook mesurent donc la **fidélité à l'enseignant**, pas la « vérité » émotionnelle. C'est exactement l'objectif : une imitation rapide des résultats de la thèse.

**Plan du notebook**

| Partie | Ce qu'on fait | Où ça tourne |
|---|---|---|
| 1 | Configuration et choix du GPU | — |
| 2 | Chargement et jointure des données | CPU |
| 3 | La couche factorielle de la thèse (rangs + poids de Bartlett) | CPU |
| 4 | Découpage entraînement / validation / test | CPU |
| 5 | Tokenisation et longueur des textes | CPU |
| 6 | Construction de l'élève et fonction de perte | GPU |
| 7 | Entraînement | GPU (≈ 1 à quelques heures) |
| 8 | Évaluation finale sur le test | GPU |
| 9 | Export pour le CPU (ONNX + int8) et mesure de vitesse | CPU |
| 10 | Fonction de prédiction pour la production | CPU |

**Prérequis** (dans l'environnement du projet, pas celui de vLLM) :
```bash
uv add transformers onnx onnxruntime scikit-learn matplotlib tqdm sentencepiece protobuf
```

## 1. Configuration

Tous les réglages sont ici, pour pouvoir relancer une expérience en ne changeant qu'une cellule.

**Le choix du GPU se fait avant d'importer `torch`.** La cellule suivante interroge `nvidia-smi`, écarte les GPU occupés ou exclus, et retient celui qui a le plus de mémoire libre. `CUDA_VISIBLE_DEVICES` ne rend ensuite visible que ce GPU : pour PyTorch, il s'appellera `cuda:0`.

Pour changer de GPU, il faut **redémarrer le noyau** : `CUDA_VISIBLE_DEVICES` n'est pris en compte qu'avant le premier usage de CUDA.

In [ ]:
import os
import subprocess

GPU_EXCLUS      = []   # GPU à ne jamais prendre, ex. [1] si un collègue l'utilise en continu
MEMOIRE_MIN_GO  = 30   # mémoire libre minimale pour entraîner l'élève
UTILISATION_MAX = 10   # au-delà de ce % d'utilisation, le GPU est considéré comme occupé


def etat_gpus():
    sortie = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,memory.used,memory.total,utilization.gpu",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=True).stdout
    gpus = []
    for ligne in sortie.strip().splitlines():
        index, utilise, total, utilisation = [x.strip() for x in ligne.split(",")]
        gpus.append({"index": int(index),
                     "libre_go": (int(total) - int(utilise)) / 1024,
                     "utilisation": int(utilisation) if utilisation.isdigit() else 100})
    return gpus


gpus = etat_gpus()
for g in gpus:
    g["disponible"] = (g["index"] not in GPU_EXCLUS and g["libre_go"] >= MEMOIRE_MIN_GO
                       and g["utilisation"] <= UTILISATION_MAX)
    print(f"GPU {g['index']} : {g['libre_go']:5.1f} Go libres, {g['utilisation']:3d} % utilisé"
          f"  -> {'disponible' if g['disponible'] else 'occupé'}")

candidats = [g for g in gpus if g["disponible"]]
if not candidats:
    raise RuntimeError("Aucun GPU disponible : réessayer plus tard ou assouplir les seuils.")
GPU_ID = max(candidats, key=lambda g: g["libre_go"])["index"]
print(f"\nGPU retenu : {GPU_ID}")

os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = os.path.abspath("../cache/")   # même cache Hugging Face que les autres notebooks

In [ ]:
import copy
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import cohen_kappa_score
from tqdm.auto import tqdm
from IPython.display import display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Appareil :", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "pas de GPU")

### Paramètres

Quelques explications sur les choix :

- **`FACTEURS`** : les fichiers de l'EFA nomment les facteurs MR1 à MR5. Ce dictionnaire leur donne le nom des colonnes de `Zscores_dimensions.csv`. La correspondance a été vérifiée : avec elle, le calcul reproduit vos Z-scores exactement.
- **`STUDENT_MODEL`** : l'encodeur de l'élève. L'enseignant est un DeBERTa-v3 **anglais** appliqué à des textes français. L'élève, lui, doit bien « lire » le français pour retrouver ce que l'enseignant répond : on part donc d'un modèle français, **CamemBERTa-v2** (même architecture que DeBERTa). Alternative à tester ensuite : `microsoft/mdeberta-v3-base` (multilingue).
- **`STUDENT_MODEL`, deuxième option** : l'encodeur de l'enseignant lui-même. L'élève « lit » alors le texte exactement comme l'enseignant, avec le même tokenizer et les mêmes représentations, et n'a plus qu'à répondre aux 28 questions en une seule lecture. C'est la piste à essayer si l'élève CamemBERTa plafonne. Le modèle est 3 fois plus gros : entraînement et production environ 3 fois plus lents, ce qui reste une seule lecture au lieu de 28.
- **`LAMBDA_FACTEURS` / `LAMBDA_LOGITS`** : poids des différentes parties de la perte (voir partie 6).
- **`LR_ENCODEUR` / `LR_TETE`** : taux d'apprentissage, c'est-à-dire la taille des corrections à chaque étape. L'encodeur est déjà pré-entraîné, on le corrige doucement. La tête part de zéro, on la corrige plus fort.
- **`EVAL_EVERY` / `PATIENCE`** : on évalue sur la validation tous les `EVAL_EVERY` pas, et on s'arrête si aucun progrès n'apparaît après `PATIENCE` évaluations consécutives (*early stopping*).

In [ ]:
# ------------------------------------------------------------- Données
DATA_DIR       = Path("../data")
EMOTIONS_PATH  = DATA_DIR / "xp_emotions_28.csv"        # 28 probabilités du zero-shot + date de publication
VERBATIM_PATH  = DATA_DIR / "verbatim_2026_07_31.csv"   # textes (Titre + Description)
ZSCORES_PATH   = DATA_DIR / "Zscores_dimensions.csv"    # 5 scores factoriels de la thèse
PATTERN_PATH   = DATA_DIR / "pattern_matrix.csv"        # saturations de l'EFA (matrice pattern, rotation oblique)
STRUCTURE_PATH = DATA_DIR / "structure_finale.csv"      # unicités (colonne u2)
ID       = "id_experience"                              # clé de jointure commune
COL_DATE = "date_publication"
FACTEURS = {"MR1": "satisfaction", "MR2": "perplexite", "MR3": "desillusion",
            "MR4": "contestation", "MR5": "detresse"}

# ------------------------------------------------------------- Élève
STUDENT_MODEL = "almanach/camembertav2-base"
# STUDENT_MODEL = "MoritzLaurer/deberta-v3-large-zeroshot-v2.0"   # option : l'encodeur de l'enseignant

# ------------------------------------------------------------- Entraînement
SEED            = 42
TEST_FRAC       = 0.10    # les 10 % de verbatims les PLUS RÉCENTS
VAL_FRAC        = 0.10
BATCH_SIZE      = 32
EPOCHS          = 6
LR_ENCODEUR     = 1e-5 if "large" in STUDENT_MODEL else 3e-5   # plus doux pour un gros modèle
LR_TETE         = 1e-3
WEIGHT_DECAY    = 0.01
WARMUP_FRAC     = 0.06
LAMBDA_FACTEURS = 1.0     # perte sur les 5 dimensions
LAMBDA_LOGITS   = 0.01    # petit terme sur les logits (voir partie 6)
EVAL_EVERY      = 1000    # pas d'entraînement entre deux évaluations
PATIENCE        = 4
N_NOEUDS        = 2001    # précision de la couche factorielle utilisée pendant l'entraînement (partie 3)

# ------------------------------------------------------------- Production
N_THREADS_CPU = 4         # nombre de cœurs CPU de la machine de production (pour la mesure de vitesse)

OUT_DIR = Path("models/eleve_emotions")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def fixer_graine(seed):
    """Rend l'expérience reproductible : mêmes tirages aléatoires à chaque exécution."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


fixer_graine(SEED)

## 2. Chargement et jointure des données

On lit trois fichiers qui partagent la même clé, l'ID de l'expérience. La fonction `lire_csv` :
- reconnaît la colonne d'ID même si elle s'appelle « ID expérience » dans un fichier et `id_experience` dans un autre ;
- convertit l'ID en texte, pour qu'un `12345` lu comme nombre dans un fichier et comme texte dans un autre se rejoignent bien ;
- supprime les doublons d'ID en le signalant.

In [ ]:
NOMS_ID = {"id_experience", "id expérience", "id experience", "idxp", "id_xp", "id"}


def lire_csv(path, sep=None):
    kw = dict(sep=sep, encoding="utf-8-sig")
    if sep is None:          # séparateur détecté automatiquement (, ou ;)
        kw["engine"] = "python"
    df = pd.read_csv(path, **kw)
    df = df.rename(columns={c: ID for c in df.columns if c.strip().lower() in NOMS_ID})
    if ID not in df.columns:
        raise KeyError(f"{path.name} : colonne d'ID introuvable. Colonnes : {list(df.columns)}")
    ids = df[ID]
    if pd.api.types.is_float_dtype(ids):      # 12345.0 -> 12345
        ids = ids.astype("Int64")
    df[ID] = ids.astype(str).str.strip()
    n = len(df)
    df = df.drop_duplicates(ID)
    print(f"{path.name:<28} {n:>8} lignes", f"({n - len(df)} doublons d'ID supprimés)" if n > len(df) else "")
    return df


emotions_df = lire_csv(EMOTIONS_PATH, sep=",")
verbatim_df = lire_csv(VERBATIM_PATH, sep=";")
z_df        = lire_csv(ZSCORES_PATH, sep=",")

### Vérifications

On contrôle que le fichier d'émotions contient bien **28 colonnes de probabilités entre 0 et 1**, et que le fichier de Z-scores contient les 5 dimensions nommées dans `FACTEURS`. Si une assertion échoue, le message indique quoi corriger.

**Point important : le texte.** L'élève doit lire **le même texte que celui que le zero-shot a lu**. On reconstruit ici `Titre + saut de ligne + Description`, comme dans les autres notebooks. Si le zero-shot n'a lu que la description, modifiez `construire_texte`.

In [ ]:
EMOTIONS = [c for c in emotions_df.columns if c != ID and pd.api.types.is_numeric_dtype(emotions_df[c])]
assert len(EMOTIONS) == 28, f"{len(EMOTIONS)} colonnes numériques trouvées au lieu de 28 : {EMOTIONS}"
mini, maxi = emotions_df[EMOTIONS].min().min(), emotions_df[EMOTIONS].max().max()
assert 0 <= mini and maxi <= 1, f"Les émotions doivent être des probabilités entre 0 et 1 (min={mini}, max={maxi})"

DIMENSIONS = list(FACTEURS.values())
manquantes = [d for d in DIMENSIONS if d not in z_df.columns]
assert not manquantes, f"Dimensions absentes de {ZSCORES_PATH.name} : {manquantes}"

emotions_df["date"] = pd.to_datetime(emotions_df[COL_DATE], errors="coerce")


def construire_texte(frame):
    return (frame["Titre"].fillna("").astype(str).str.strip() + "\n\n"
            + frame["Description"].fillna("").astype(str).str.strip()).str.strip()


verbatim_df["texte"] = construire_texte(verbatim_df)

print("Émotions   :", ", ".join(EMOTIONS))
print("Dimensions :", ", ".join(DIMENSIONS))
print(f"Dates      : du {emotions_df['date'].min():%d/%m/%Y} au {emotions_df['date'].max():%d/%m/%Y}, "
      f"{emotions_df['date'].isna().sum()} non lues")

### Jointure

Deux tableaux sortent de cette étape :
- **`ref`** : les verbatims qui ont à la fois des émotions et des Z-scores. C'est **l'échantillon de l'EFA de la thèse**. Il sert de référence figée pour calculer les rangs (partie 3).
- **`df`** : les mêmes verbatims, avec leur texte. C'est sur eux qu'on entraîne et qu'on évalue l'élève.

Le décompte à chaque étape permet de repérer une perte anormale, par exemple un fichier qui n'a pas le même format d'ID.

In [ ]:
ref = emotions_df[[ID, "date"] + EMOTIONS].merge(z_df[[ID] + DIMENSIONS], on=ID, how="inner")
ref = ref.dropna(subset=EMOTIONS + DIMENSIONS).reset_index(drop=True)
print(f"{'émotions + Z-scores (réf.)':<28} {len(ref):>8}")

df = ref.merge(verbatim_df[[ID, "texte"]], on=ID, how="inner")
df = df[df["texte"].str.len() > 0].reset_index(drop=True)
print(f"{'+ textes non vides':<28} {len(df):>8}")
df.head(3)

## 3. La couche factorielle de la thèse

### Le calcul de la thèse

Les distributions des probabilités du zero-shot sont très asymétriques : la plupart des valeurs sont collées à 0. L'EFA a donc été faite sur les **rangs** des probabilités. Pour chaque verbatim, le score d'une dimension s'obtient en quatre temps :

1. **Rang.** Chaque probabilité est remplacée par son rang dans le corpus de référence (ex-aequo : rang moyen).
2. **Standardisation des rangs.** $r_j = (\text{rang}_j - \text{moyenne}_j) / \text{écart-type}_j$.
3. **Poids de Bartlett.** $\text{score}_k = \sum_j r_j \, W_{jk}$, avec
$$W = \Psi^{-1}\Lambda\,(\Lambda^\top \Psi^{-1} \Lambda)^{-1}$$
où $\Lambda$ est la matrice *pattern* des saturations (`pattern_matrix.csv`) et $\Psi$ les unicités (`u2` dans `structure_finale.csv`).
4. **Re-standardisation.** Chaque score est ramené à une moyenne de 0 et un écart-type de 1 sur le corpus de référence.

### Pourquoi les rangs changent la façon d'entraîner l'élève

Le passage aux rangs amplifie énormément les faibles probabilités. Pour la colère par exemple, passer de 0,001 à 0,01 fait gagner environ 8 centiles, presque autant que passer de 0,1 à 0,5. Une erreur de l'élève qui paraît minuscule sur la probabilité peut donc décaler un score factoriel dès la première décimale.

C'est pourquoi l'élève travaille sur les **logits** : $\text{logit}(p) = \log\frac{p}{1-p}$. Le logit est une autre façon d'écrire une probabilité, qui étire les valeurs proches de 0 et de 1 : 0,001 devient −6,9 et 0,01 devient −4,6. Les écarts qui comptent pour les rangs deviennent ainsi visibles.

### Pour un nouveau verbatim

On le situe par rapport aux verbatims de la thèse. Le corpus de référence est **figé** : on ne recalcule jamais les rangs en ajoutant les nouveaux verbatims, sinon tous les anciens scores bougeraient. Concrètement, on garde pour chaque émotion une **table** qui associe à chaque valeur observée dans le corpus son rang standardisé, et on interpole pour les valeurs nouvelles. Au-delà de la valeur la plus extrême du corpus, le rang plafonne.

In [ ]:
# --- Poids de Bartlett à partir des résultats de l'EFA
pattern = pd.read_csv(PATTERN_PATH, index_col=0, encoding="utf-8-sig").rename(columns=FACTEURS)
unicites = pd.read_csv(STRUCTURE_PATH, encoding="utf-8-sig").set_index("Item")["u2"]

manquantes = [e for e in EMOTIONS if e not in pattern.index or e not in unicites.index]
assert not manquantes, f"Émotions absentes des fichiers de l'EFA : {manquantes}"
L = pattern.loc[EMOTIONS, DIMENSIONS].to_numpy(np.float64)          # 28 × 5
psi_inv = 1 / unicites.loc[EMOTIONS].to_numpy(np.float64)            # 28
W = (psi_inv[:, None] * L) @ np.linalg.inv(L.T @ (psi_inv[:, None] * L))

EPS = 1e-6


def logit(p):
    p = np.clip(np.asarray(p, dtype=np.float64), EPS, 1 - EPS)
    return np.log(p / (1 - p))


# --- Table de référence figée : valeur (en logit) -> rang standardisé, pour chaque émotion
TABLE = {}
for e in EMOTIONS:
    rangs = ref[e].rank()                                    # ex-aequo : rang moyen
    r_std = (rangs - rangs.mean()) / rangs.std()
    par_valeur = r_std.groupby(ref[e]).mean()                # une ligne par valeur distincte, triée
    TABLE[e] = (logit(par_valeur.index.to_numpy()), par_valeur.to_numpy(np.float64))


def rangs_standardises(logits):
    """(n, 28) logits -> (n, 28) rangs standardisés, par interpolation dans la table figée."""
    logits = np.asarray(logits, dtype=np.float64)
    return np.column_stack([np.interp(logits[:, j], *TABLE[e]) for j, e in enumerate(EMOTIONS)])


# --- Re-standardisation finale des scores, figée sur le corpus de référence
F_ref = rangs_standardises(logit(ref[EMOTIONS])) @ W
W_FINAL = W / F_ref.std(0, ddof=1)
C_FINAL = -F_ref.mean(0) / F_ref.std(0, ddof=1)


def couche_factorielle(logits):
    """Le calcul complet de la thèse : 28 logits -> 5 scores factoriels."""
    return rangs_standardises(logits) @ W_FINAL + C_FINAL


print("Nombre de valeurs distinctes par émotion :",
      min(len(t[0]) for t in TABLE.values()), "à", max(len(t[0]) for t in TABLE.values()))

### Le contrôle

On recalcule les scores de tous les verbatims de référence et on les compare à vos Z-scores. L'écart maximal doit être de l'ordre de 0,00001 : il ne vient que de l'arrondi des saturations dans `pattern_matrix.csv`. Un écart plus grand signale une différence de méthode (correspondance MR ↔ dimension, échantillon de l'EFA…), et il faut la comprendre avant d'aller plus loin.

In [ ]:
Z_ref = ref[DIMENSIONS].to_numpy(np.float64)
Z_calc = couche_factorielle(logit(ref[EMOTIONS]))
controle = pd.DataFrame({
    "corrélation": [np.corrcoef(Z_ref[:, k], Z_calc[:, k])[0, 1] for k in range(len(DIMENSIONS))],
    "écart moyen": np.abs(Z_ref - Z_calc).mean(0),
    "écart max": np.abs(Z_ref - Z_calc).max(0),
}, index=DIMENSIONS)
display(controle.round(6))

if controle["écart max"].max() < 1e-3:
    print("La couche factorielle reproduit exactement vos Z-scores.")
else:
    print("ATTENTION : la couche ne reproduit pas vos Z-scores. Vérifier FACTEURS et l'échantillon de référence.")

### La version de la couche utilisée pendant l'entraînement

Pendant l'entraînement, il faut faire passer la couche factorielle sur le GPU, des milliers de fois, et pouvoir calculer comment corriger l'élève à travers elle. Les tables complètes (des dizaines de milliers de valeurs par émotion) seraient trop lourdes. On les résume par `N_NOEUDS` points régulièrement espacés en rang, et on interpole linéairement entre eux.

Cette version approchée ne sert **que** pour la perte d'entraînement. Toutes les évaluations et la production utilisent la table complète, exacte. On vérifie quand même que l'approximation reste très fine.

In [ ]:
X_NOEUDS = np.zeros((len(EMOTIONS), N_NOEUDS))
Y_NOEUDS = np.zeros((len(EMOTIONS), N_NOEUDS))
for j, e in enumerate(EMOTIONS):
    x, y = TABLE[e]
    Y_NOEUDS[j] = np.linspace(y.min(), y.max(), N_NOEUDS)
    X_NOEUDS[j] = np.interp(Y_NOEUDS[j], y, x)      # le rang croît avec la valeur : on peut inverser


def couche_approchee(logits):
    logits = np.asarray(logits, dtype=np.float64)
    r = np.column_stack([np.interp(logits[:, j], X_NOEUDS[j], Y_NOEUDS[j]) for j in range(len(EMOTIONS))])
    return r @ W_FINAL + C_FINAL


echantillon_ref = logit(ref[EMOTIONS].sample(20_000, random_state=SEED))
ecart = np.abs(couche_approchee(echantillon_ref) - couche_factorielle(echantillon_ref))
print(f"Écart entre la couche approchée et la couche exacte : moyen {ecart.mean():.5f}, max {ecart.max():.4f}")

### Lire les poids

La carte montre les poids de Bartlett $W$ : ceux de la thèse, appliqués aux rangs standardisés. Vous devez y retrouver la structure de votre EFA, avec les émotions qui saturent fortement une dimension portant les plus gros poids.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 9))
lim = np.abs(W).max()
im = ax.imshow(W, cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto")
ax.set_xticks(range(len(DIMENSIONS)), DIMENSIONS, rotation=45, ha="right")
ax.set_yticks(range(len(EMOTIONS)), EMOTIONS)
plt.colorbar(im, ax=ax, label="poids de Bartlett")
ax.set_title("Poids de Bartlett de la thèse (W)")
plt.tight_layout()
plt.show()

np.savez(OUT_DIR / "couche_factorielle.npz", W_final=W_FINAL, c_final=C_FINAL,
         **{f"x_{j}": TABLE[e][0] for j, e in enumerate(EMOTIONS)},
         **{f"y_{j}": TABLE[e][1] for j, e in enumerate(EMOTIONS)})
print("Couche factorielle enregistrée dans", OUT_DIR / "couche_factorielle.npz")

## 4. Découpage entraînement / validation / test

- **Test = les 10 % les plus récents.** On évalue l'élève sur des verbatims postérieurs à tout ce qu'il a vu, comme ce sera le cas en production. Si les avis évoluent dans le temps, ce test le révélera.
- **Validation = 10 % tirés au hasard parmi le reste.** Elle sert à surveiller l'entraînement et à choisir le meilleur moment pour s'arrêter.
- **Entraînement = tout le reste.**

Le jeu de test n'est utilisé **qu'une seule fois**, à la fin. Si on le consultait pour régler des paramètres, on finirait par ajuster le modèle à ce jeu précis, et la note finale serait trop optimiste.

Les rangs de la couche factorielle restent calculés sur **tout** le corpus de référence, y compris le test. Ce n'est pas une fuite d'information vers l'élève : la couche est la définition fixe des dimensions de la thèse, pas quelque chose que l'élève apprend.

In [ ]:
datees = df[df["date"].notna()].sort_values("date")
non_datees = df[df["date"].isna()]

n_test = int(len(df) * TEST_FRAC)
test_df = datees.iloc[-n_test:]
reste = pd.concat([datees.iloc[:-n_test], non_datees]).sample(frac=1, random_state=SEED)
n_val = int(len(df) * VAL_FRAC)
val_df = reste.iloc[:n_val]
train_df = reste.iloc[n_val:]

for nom, part in [("entraînement", train_df), ("validation", val_df), ("test", test_df)]:
    print(f"{nom:<13} {len(part):>7} verbatims | du {part['date'].min():%d/%m/%Y} au {part['date'].max():%d/%m/%Y}")


def logits_enseignant(frame):
    return logit(frame[EMOTIONS])


def scores_enseignant(frame):
    return frame[DIMENSIONS].to_numpy(np.float64)     # vos Z-scores

## 5. Tokenisation et longueur des textes

Un modèle de langue ne lit pas des lettres mais des **tokens** : des morceaux de mots tirés d'un vocabulaire fixe (« administration » peut devenir `▁administr` + `ation`). Le **tokenizer** fait ce découpage et remplace chaque token par son numéro dans le vocabulaire.

Le temps de calcul augmente avec le nombre de tokens, et le modèle a une longueur maximale (512 tokens). Les avis étant limités à 2 000 caractères, on règle `MAX_LEN` pour couvrir **tous** les verbatims, dans la limite de 512 : l'élève lit ainsi la même chose que l'enseignant.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)

exemple = train_df["texte"].iloc[0]
print("Texte  :", exemple[:150].replace("\n", " "), "...")
print("Tokens :", tokenizer.tokenize(exemple)[:25], "...")
print("Numéros:", tokenizer(exemple)["input_ids"][:25], "...")

In [ ]:
echantillon = train_df["texte"].sample(min(10_000, len(train_df)), random_state=SEED).tolist()
longueurs = np.array([len(ids) for ids in tokenizer(echantillon)["input_ids"]])

for q in [50, 90, 95, 99, 100]:
    print(f"{q:>3} % des verbatims font moins de {np.percentile(longueurs, q):>5.0f} tokens")

MAX_LEN = int(min(512, np.ceil(longueurs.max() / 32) * 32))
print(f"\nMAX_LEN = {MAX_LEN} tokens ({(longueurs > MAX_LEN).mean():.1%} des verbatims seront tronqués)")

plt.figure(figsize=(7, 3))
plt.hist(np.minimum(longueurs, 600), bins=60)
plt.axvline(MAX_LEN, color="red", label=f"MAX_LEN = {MAX_LEN}")
plt.xlabel("nombre de tokens"); plt.ylabel("verbatims"); plt.legend()
plt.show()

### Les données en paquets (*batches*)

Le GPU traite les verbatims par paquets de `BATCH_SIZE`. Dans un paquet, les textes doivent avoir la même longueur : on complète les plus courts avec un token vide (*padding*). Le **masque d'attention** (`attention_mask`) vaut 1 sur les vrais tokens et 0 sur le padding, pour que le modèle ignore ce dernier.

On complète seulement jusqu'au texte le plus long **du paquet** (*padding dynamique*), pas jusqu'à `MAX_LEN` : c'est nettement plus rapide quand la plupart des textes sont courts.

Les **cibles** sont les 28 logits de l'enseignant.

In [ ]:
class VerbatimDataset(Dataset):
    def __init__(self, frame):
        self.textes = frame["texte"].tolist()
        self.cibles = logits_enseignant(frame).astype(np.float32)   # les 28 logits de l'enseignant

    def __len__(self):
        return len(self.textes)

    def __getitem__(self, i):
        return self.textes[i], self.cibles[i]


def assembler(batch):
    textes, cibles = zip(*batch)
    enc = tokenizer(list(textes), truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"],
            "cibles": torch.from_numpy(np.stack(cibles))}


train_loader = DataLoader(VerbatimDataset(train_df), batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=assembler, num_workers=4, pin_memory=True)

paquet = next(iter(train_loader))
for k, v in paquet.items():
    print(f"{k:<15} {tuple(v.shape)}")

## 6. L'élève

L'élève a trois étages, plus la couche figée.

**a) L'encodeur (le « lecteur »).** Il transforme chaque token en un vecteur de 768 nombres qui représente son sens *dans le contexte* de la phrase. Il est pré-entraîné sur des milliards de mots de français : il « sait lire », il reste à lui apprendre ce que l'enseignant répond.

**b) Le résumé du texte (*mean pooling*).** On fait la moyenne des vecteurs de tous les vrais tokens (padding exclu grâce au masque). On obtient **un seul vecteur de 768 nombres pour tout le verbatim**.

**c) La tête (le « jury »).** Une couche linéaire transforme ce vecteur en **28 logits**, un par émotion, chacun calculé **indépendamment des autres** : c'est ce qui garantit l'évaluation indépendante des 28 émotions, comme le zero-shot en multi-label. Pour retrouver une probabilité, il suffit d'appliquer la sigmoïde : $p = 1 / (1 + e^{-\text{logit}})$.

**d) La couche factorielle figée** (version approchée de la partie 3). Elle n'est utilisée que pour la perte d'entraînement. Ses tables sont enregistrées comme `buffer` : elles font partie du modèle, mais l'entraînement ne les modifie **jamais**. Elle est calculée en précision complète (float32), car la précision réduite utilisée pour accélérer l'entraînement déformerait les scores.

**Un détail qui accélère l'entraînement.** On initialise le biais de la tête pour que l'élève non entraîné prédise d'emblée le **logit moyen** de chaque émotion. Il part ainsi d'une réponse raisonnable au lieu de 0 partout.

In [ ]:
class Eleve(nn.Module):
    def __init__(self, nom_modele, n_emotions, x_noeuds, y_noeuds, w_final, c_final):
        super().__init__()
        self.encodeur = AutoModel.from_pretrained(nom_modele)
        h = self.encodeur.config.hidden_size
        self.tete = nn.Sequential(nn.Dropout(0.1), nn.Linear(h, n_emotions))
        # Couche factorielle figée (version approchée, pour la perte d'entraînement)
        self.register_buffer("xn", torch.tensor(x_noeuds, dtype=torch.float32))
        self.register_buffer("yn", torch.tensor(y_noeuds, dtype=torch.float32))
        self.register_buffer("W", torch.tensor(w_final, dtype=torch.float32))
        self.register_buffer("c", torch.tensor(c_final, dtype=torch.float32))

    def forward(self, input_ids, attention_mask):
        etats = self.encodeur(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        masque = attention_mask.unsqueeze(-1).to(etats.dtype)
        resume = (etats * masque).sum(1) / masque.sum(1).clamp(min=1e-6)       # mean pooling
        return self.tete(resume).float()                                       # 28 logits

    def rangs(self, logits):
        """Logits (lot × 28) -> rangs standardisés (lot × 28), par interpolation dans les tables figées."""
        with torch.autocast(device_type=logits.device.type, enabled=False):
            v = logits.float().T                                               # 28 × lot
            k = self.xn.shape[1]
            i = (v.unsqueeze(-1) > self.xn.unsqueeze(1)).sum(-1).clamp(1, k - 1)   # nœud juste au-dessus
            x0, x1 = self.xn.gather(1, i - 1), self.xn.gather(1, i)
            y0, y1 = self.yn.gather(1, i - 1), self.yn.gather(1, i)
            poids = ((v - x0) / (x1 - x0).clamp(min=1e-12)).clamp(0, 1)        # au-delà des bornes : plafond
            return (y0 + poids * (y1 - y0)).T                                  # lot × 28

    def facteurs(self, logits):
        """Logits (lot × 28) -> 5 scores factoriels."""
        return self.rangs(logits) @ self.W + self.c


model = Eleve(STUDENT_MODEL, len(EMOTIONS), X_NOEUDS, Y_NOEUDS, W_FINAL, C_FINAL).to(device)

with torch.no_grad():
    model.tete[1].bias.copy_(torch.tensor(logits_enseignant(train_df).mean(0), dtype=torch.float32))
    model.tete[1].weight.mul_(0.1)

n_param = sum(p.numel() for p in model.parameters())
print(f"Paramètres de l'élève : {n_param / 1e6:.0f} millions (enseignant : ~435 millions, lu 28 fois)")

### Ce qui se passe à l'intérieur, sur un paquet

Suivons les dimensions des tenseurs à chaque étage, pour le paquet chargé plus haut.

In [ ]:
model.eval()
with torch.no_grad():
    ids, masque = paquet["input_ids"].to(device), paquet["attention_mask"].to(device)
    etats = model.encodeur(input_ids=ids, attention_mask=masque).last_hidden_state
    logits = model(ids, masque)
    facteurs = model.facteurs(logits)
model.train()

print(f"tokens en entrée     {tuple(ids.shape)}        (verbatims × tokens)")
print(f"sortie de l'encodeur {tuple(etats.shape)}   (verbatims × tokens × 768)")
print(f"logits               {tuple(logits.shape)}         (verbatims × 28 émotions)")
print(f"scores factoriels    {tuple(facteurs.shape)}          (verbatims × 5 dimensions)")
print("\nPremier verbatim, avant entraînement (probabilités) :")
print(pd.DataFrame({"élève": torch.sigmoid(logits[0]).cpu().numpy(),
                    "enseignant": torch.sigmoid(paquet["cibles"][0]).numpy()},
                   index=EMOTIONS).round(3).T)

### La fonction de perte : mesurer l'écart avec l'enseignant

À chaque pas, on mesure à quel point l'élève s'écarte de l'enseignant. C'est la **perte** : plus elle est basse, meilleure est l'imitation. Elle a trois parties.

1. **Perte sur les rangs des 28 émotions** (la principale). On convertit les logits de l'élève et ceux de l'enseignant en **rangs standardisés**, avec la couche figée, et on mesure l'erreur quadratique moyenne entre les deux. C'est exactement la mesure de la thèse : un écart compte en proportion de son effet sur le rang. Une erreur entre 0,001 et 0,01 pèse lourd, parce qu'elle fait bouger le rang de plusieurs centiles. Une erreur entre deux valeurs quasi nulles qui tombent dans le même paquet d'ex-aequo ne pèse presque rien, parce que leur rang est le même.
2. **Perte sur les 5 dimensions** (`LAMBDA_FACTEURS`) : l'écart entre les scores factoriels de l'élève et ceux de l'enseignant. Elle pousse l'élève à soigner surtout les émotions qui pèsent lourd dans vos dimensions.
3. **Petit terme sur les logits** (`LAMBDA_LOGITS`, très faible). Au-delà des valeurs extrêmes du corpus, le rang plafonne : si l'élève y tombe, la perte sur les rangs ne lui indique plus dans quelle direction se corriger. Ce terme le ramène vers les bonnes valeurs dans ces cas-là, sans peser sur le reste.

**Pourquoi ce changement par rapport au premier essai.** Le premier essai comparait directement les logits. Cette mesure accorde énormément d'importance aux valeurs extrêmes : prédire −6 quand l'enseignant dit −10 coûtait 16, alors que ces deux valeurs ont souvent presque le même rang. L'élève dépensait ses efforts sur des écarts sans effet sur vos scores.

Une perte de 0 sur les rangs signifierait une imitation parfaite. Une partie ne descendra jamais à 0 : l'enseignant a une part de bruit dans les très faibles probabilités, que l'élève ne peut pas deviner à partir du texte.

In [ ]:
def calculer_perte(logits, cibles):
    rangs_eleve, rangs_ens = model.rangs(logits), model.rangs(cibles)
    perte_rangs = F.mse_loss(rangs_eleve, rangs_ens)
    perte_dim = F.mse_loss(rangs_eleve @ model.W + model.c, rangs_ens @ model.W + model.c)
    perte_logits = F.mse_loss(logits, cibles)
    totale = perte_rangs + LAMBDA_FACTEURS * perte_dim + LAMBDA_LOGITS * perte_logits
    return totale, {"rangs": perte_rangs.item(), "dimensions": perte_dim.item(), "logits": perte_logits.item()}

### L'optimiseur et le calendrier du taux d'apprentissage

- **AdamW** ajuste chaque paramètre dans la direction qui fait baisser la perte. Le *weight decay* freine légèrement les poids trop grands, ce qui limite le sur-apprentissage.
- **Deux taux d'apprentissage** : petit pour l'encodeur (on ne veut pas abîmer ce qu'il sait déjà), plus grand pour la tête (elle part de zéro).
- **Échauffement puis décroissance** : le taux monte progressivement pendant les premiers 6 % des pas (les premières corrections, encore très imprécises, ne doivent pas tout bouleverser), puis il décroît linéairement jusqu'à 0.

In [ ]:
sans_decay = ("bias", "LayerNorm.weight", "layernorm.weight", "norm.weight")
groupes = [
    {"params": [p for n, p in model.encodeur.named_parameters() if not any(s in n for s in sans_decay)],
     "lr": LR_ENCODEUR, "weight_decay": WEIGHT_DECAY},
    {"params": [p for n, p in model.encodeur.named_parameters() if any(s in n for s in sans_decay)],
     "lr": LR_ENCODEUR, "weight_decay": 0.0},
    {"params": model.tete.parameters(), "lr": LR_TETE, "weight_decay": 0.0},
]
optimizer = torch.optim.AdamW(groupes)

n_pas = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_FRAC * n_pas), n_pas)
print(f"{len(train_loader)} pas par epoch, {n_pas} pas au maximum, "
      f"une évaluation tous les {EVAL_EVERY} pas")

### Prédire et mesurer la fidélité

`predire` fait passer un jeu de données dans l'élève, sans calculer de corrections (`torch.no_grad`), et renvoie les 28 logits. Les 5 scores sont ensuite calculés avec la **couche exacte** de la partie 3.

Deux mesures :
- **Sur les 5 dimensions** : la **corrélation de Pearson** entre les scores de l'élève et vos Z-scores. Elle vaut 1 si l'élève classe et espace les verbatims exactement comme la thèse. Repères : **au-dessus de 0,90 c'est bon, au-dessus de 0,95 c'est excellent**.
- **Sur les 28 émotions** : la corrélation entre les **rangs standardisés** de l'élève et ceux de l'enseignant. C'est ce qui compte pour vos dimensions : une émotion peut sembler bien imitée en probabilités tout en étant mal classée dans sa partie basse.

In [ ]:
@torch.no_grad()
def predire(model, frame, batch_size=128):
    model.eval()
    loader = DataLoader(VerbatimDataset(frame), batch_size=batch_size, shuffle=False,
                        collate_fn=assembler, num_workers=4)
    sorties = []
    for b in loader:
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
            sorties.append(model(b["input_ids"].to(device), b["attention_mask"].to(device)).float().cpu())
    model.train()
    return torch.cat(sorties).numpy()


def correlations(a, b):
    """Corrélation de Pearson colonne par colonne entre deux matrices de même forme."""
    a = a - a.mean(0)
    b = b - b.mean(0)
    return (a * b).sum(0) / np.sqrt((a ** 2).sum(0) * (b ** 2).sum(0))


R_val = rangs_standardises(logits_enseignant(val_df))
Z_val = scores_enseignant(val_df)

**Point de départ.** Avant tout entraînement, l'élève prédit à peu près la moyenne pour tout le monde : les corrélations doivent être proches de 0. C'est la référence à partir de laquelle on mesure les progrès.

In [ ]:
Lv = predire(model, val_df)
print("Corrélations avant entraînement :")
print(pd.Series(correlations(couche_factorielle(Lv), Z_val), index=DIMENSIONS).round(3).to_string())

## 7. Entraînement

À chaque pas :
1. on charge un paquet de 32 verbatims sur le GPU ;
2. l'élève prédit (*forward*), en précision réduite **bfloat16** pour aller environ deux fois plus vite ;
3. on calcule la perte ;
4. on calcule dans quelle direction modifier chaque paramètre pour la réduire (*backward*) ;
5. on limite la taille de la correction (*gradient clipping*), par sécurité contre les pas aberrants ;
6. l'optimiseur applique la correction.

Tous les `EVAL_EVERY` pas, on mesure la fidélité sur la validation. **Si elle s'améliore, on sauvegarde le modèle.** Sinon, on compte un échec, et on arrête après `PATIENCE` échecs consécutifs.

Comptez environ 30 minutes par epoch sur une H100 libre avec CamemBERTa-v2, soit 3 heures pour 6 epochs, et environ 3 fois plus avec l'encodeur de l'enseignant. La barre de progression affiche la perte moyenne récente : elle doit baisser rapidement au début, puis plus lentement.

**Si le noyau plante** : le meilleur modèle est déjà sauvegardé dans `models/eleve_emotions/meilleur.pt`. Après avoir relancé les parties 1 à 6, vous pouvez passer directement à la partie 8.

In [ ]:
historique = []
meilleur_score, echecs, pas, arret = -np.inf, 0, 0, False
PARTIES = ["totale", "rangs", "dimensions", "logits"]
cumul = dict.fromkeys(PARTIES, 0.0) | {"n": 0}
debut = time.time()

model.train()
for epoch in range(EPOCHS):
    barre = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}")
    for batch in barre:
        ids = batch["input_ids"].to(device, non_blocking=True)
        masque = batch["attention_mask"].to(device, non_blocking=True)
        cibles = batch["cibles"].to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
            logits = model(ids, masque)
        perte, parties = calculer_perte(logits, cibles)

        optimizer.zero_grad(set_to_none=True)
        perte.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        pas += 1

        cumul["totale"] += perte.item(); cumul["n"] += 1
        for k, v in parties.items():
            cumul[k] += v
        if pas % 50 == 0:
            barre.set_postfix(perte=f"{cumul['totale'] / cumul['n']:.4f}")

        if pas % EVAL_EVERY == 0 or pas == n_pas:
            Lv = predire(model, val_df)
            r_dim = correlations(couche_factorielle(Lv), Z_val)
            r_emo = correlations(rangs_standardises(Lv), R_val)
            ligne = {"pas": pas, "epoch": epoch + 1,
                     **{f"perte_{k}": cumul[k] / cumul["n"] for k in PARTIES},
                     "r_dimensions_moyen": r_dim.mean(), "r_emotions_moyen": r_emo.mean(),
                     **{f"r_{d}": r for d, r in zip(DIMENSIONS, r_dim)}}
            historique.append(ligne)
            cumul = dict.fromkeys(PARTIES, 0.0) | {"n": 0}

            if r_dim.mean() > meilleur_score:
                meilleur_score, echecs = r_dim.mean(), 0
                torch.save(model.state_dict(), OUT_DIR / "meilleur.pt")
                statut = "nouveau meilleur, sauvegardé"
            else:
                echecs += 1
                statut = f"pas de progrès ({echecs}/{PATIENCE})"
            print(f"pas {pas:>6} | perte {ligne['perte_totale']:.4f} | r dimensions {r_dim.mean():.4f} "
                  f"| r émotions (rangs) {r_emo.mean():.4f} | {statut} | {(time.time() - debut) / 60:.0f} min")
            pd.DataFrame(historique).to_csv(OUT_DIR / "historique.csv", index=False)

            if echecs >= PATIENCE:
                arret = True
                break
    if arret:
        print("Arrêt anticipé : la validation ne progresse plus.")
        break

print(f"Terminé en {(time.time() - debut) / 60:.0f} min. Meilleure corrélation moyenne : {meilleur_score:.4f}")

### Lire les courbes

- **À gauche, les trois parties de la perte**, en échelle logarithmique pour qu'on les voie toutes malgré leurs ordres de grandeur différents. Elles doivent baisser vite puis se stabiliser.
- **À droite, la fidélité sur la validation** doit monter puis plafonner.
- **Signal de sur-apprentissage** : la perte d'entraînement continue de baisser alors que la validation stagne ou redescend. L'élève apprend alors par cœur les exemples d'entraînement au lieu d'apprendre la règle. L'arrêt anticipé et la sauvegarde du meilleur modèle vous protègent de ce cas.

In [ ]:
h = pd.DataFrame(historique) if historique else pd.read_csv(OUT_DIR / "historique.csv")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
for k, nom in [("rangs", "rangs des émotions"), ("dimensions", "dimensions"), ("logits", "logits (petit terme)")]:
    ax1.plot(h["pas"], h[f"perte_{k}"], label=nom)
ax1.set_yscale("log")
ax1.set_xlabel("pas"); ax1.set_title("Perte d'entraînement (échelle log)"); ax1.legend()
for d in DIMENSIONS:
    ax2.plot(h["pas"], h[f"r_{d}"], label=d)
ax2.axhline(0.95, color="grey", ls="--", lw=0.8)
ax2.set_xlabel("pas"); ax2.set_title("Corrélation élève / thèse (validation)"); ax2.legend()
plt.tight_layout()
plt.show()

## 8. Évaluation finale sur le test (les verbatims les plus récents)

On recharge le **meilleur** modèle sauvegardé (pas forcément le dernier), puis on mesure la fidélité à quatre niveaux.

In [ ]:
model.load_state_dict(torch.load(OUT_DIR / "meilleur.pt", map_location=device))
L_test_eleve = predire(model, test_df)
F_test_eleve = couche_factorielle(L_test_eleve)
R_test_eleve = rangs_standardises(L_test_eleve)

F_test_ens = scores_enseignant(test_df)
R_test_ens = rangs_standardises(logits_enseignant(test_df))

### Niveau 1 : les 5 dimensions (l'indicateur principal)

- **r** : corrélation de Pearson avec vos Z-scores. Visez au-dessus de 0,90, idéalement 0,95.
- **MAE** : écart absolu moyen, dans l'unité des Z-scores. Une MAE de 0,15 signifie qu'en moyenne l'élève se trompe de 0,15 écart-type.
- **à ±0,10** et **à ±0,25** : part des verbatims dont le score de l'élève tombe à moins de 0,10 (ou 0,25) du score de la thèse. C'est la mesure la plus concrète si vous utilisez les scores arrondis à 2 décimales.

In [ ]:
ecarts = np.abs(F_test_eleve - F_test_ens)
res_dim = pd.DataFrame({
    "r": correlations(F_test_eleve, F_test_ens),
    "MAE": ecarts.mean(0),
    "à ±0,10": (ecarts <= 0.10).mean(0),
    "à ±0,25": (ecarts <= 0.25).mean(0),
}, index=DIMENSIONS)
display(res_dim.style.format({"r": "{:.3f}", "MAE": "{:.3f}", "à ±0,10": "{:.0%}", "à ±0,25": "{:.0%}"}))

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for k, (ax, d) in enumerate(zip(axes, DIMENSIONS)):
    ax.hexbin(F_test_ens[:, k], F_test_eleve[:, k], gridsize=50, bins="log", cmap="Blues")
    lo, hi = np.percentile(F_test_ens[:, k], [0.5, 99.5])
    ax.plot([lo, hi], [lo, hi], "r--", lw=1)
    ax.set_title(f"{d}  (r = {res_dim.loc[d, 'r']:.3f})")
    ax.set_xlabel("thèse")
axes[0].set_ylabel("élève")
plt.tight_layout()
plt.show()

Sur ces nuages de points, chaque verbatim est un point : plus ils se serrent autour de la diagonale rouge, plus l'imitation est fidèle. Regardez surtout **les extrémités** : un élève peut être bon en moyenne mais écraser les cas extrêmes (les points s'aplatissent à l'horizontale aux bords). Ce sont souvent les verbatims qui vous intéressent le plus, par exemple une détresse forte.

### Niveau 2 : les 28 émotions

Une émotion mal reproduite n'est grave que si elle pèse lourd dans une dimension. Le tableau croise donc la corrélation de chaque émotion (sur les rangs) avec son poids maximal dans la couche factorielle.

In [ ]:
P_test_eleve = 1 / (1 + np.exp(-L_test_eleve))
res_emo = pd.DataFrame({
    "r (rangs)": correlations(R_test_eleve, R_test_ens),
    "écart moyen de proba": np.abs(P_test_eleve - test_df[EMOTIONS].to_numpy()).mean(0),
    "poids max |W|": np.abs(W).max(1),
    "dimension du plus gros poids": [DIMENSIONS[k] for k in np.abs(W).argmax(1)],
}, index=EMOTIONS).sort_values("r (rangs)")
display(res_emo.round(3))

### Niveau 3 : la structure du modèle factoriel

Avoir de bonnes corrélations dimension par dimension ne garantit pas que l'élève respecte **la structure** de votre modèle, c'est-à-dire le lien entre chaque émotion et chaque dimension.

Pour le vérifier, on calcule pour l'enseignant puis pour l'élève la **matrice de structure** : la corrélation entre chacune des 28 émotions (en rangs) et chacune des 5 dimensions. On compare ensuite les deux matrices, colonne par colonne, avec le **coefficient de congruence de Tucker** (φ). C'est la mesure classique pour comparer deux solutions factorielles : **φ ≥ 0,95** signifie que les deux facteurs sont considérés comme identiques, et entre 0,85 et 0,94 qu'ils sont proches.

On n'a pas besoin de refaire une EFA (avec ses problèmes de rotation et d'ordre des facteurs) : on compare directement les liens émotion-dimension.

In [ ]:
def matrice_structure(R, Fz):
    Rc = (R - R.mean(0)) / R.std(0)
    Fc = (Fz - Fz.mean(0)) / Fz.std(0)
    return Rc.T @ Fc / len(R)            # 28 × 5


S_ens = matrice_structure(R_test_ens, F_test_ens)
S_eleve = matrice_structure(R_test_eleve, F_test_eleve)
phi = (S_ens * S_eleve).sum(0) / np.sqrt((S_ens ** 2).sum(0) * (S_eleve ** 2).sum(0))
res_dim["φ de Tucker"] = phi
display(pd.Series(phi, index=DIMENSIONS, name="φ de Tucker").round(4).to_frame())

### Niveau 4 : la dimension dominante (si vous étiquetez chaque verbatim)

Si vous attribuez à chaque verbatim sa dimension la plus élevée, on mesure le pourcentage de verbatims où l'élève et la thèse désignent **la même**. Le **κ de Cohen** corrige ce pourcentage de l'accord dû au hasard.

Rappel : les scores sont centrés sur la moyenne du corpus. « Dimension la plus élevée » veut dire « la plus au-dessus de la moyenne », pas forcément « présente ». Les désaccords se concentrent sur les verbatims où deux dimensions sont presque à égalité, ce qui est normal.

In [ ]:
dom_ens = F_test_ens.argmax(1)
dom_eleve = F_test_eleve.argmax(1)
print(f"Même dimension dominante : {(dom_ens == dom_eleve).mean():.1%}")
print(f"κ de Cohen               : {cohen_kappa_score(dom_ens, dom_eleve):.3f}")

confusion = pd.crosstab(pd.Series(np.array(DIMENSIONS)[dom_ens], name="thèse"),
                        pd.Series(np.array(DIMENSIONS)[dom_eleve], name="élève"))
display(confusion)

res_dim.to_csv(OUT_DIR / "resultats_test_dimensions.csv")
res_emo.to_csv(OUT_DIR / "resultats_test_emotions.csv")

## 9. Export pour le CPU et mesure de vitesse

Jusqu'ici l'élève tourne en PyTorch sur GPU. Pour la production sur CPU, on fait deux transformations.

**ONNX.** On « fige » le modèle dans un format standard, optimisé pour l'exécution seule : plus besoin de PyTorch en production, seulement `onnxruntime`, beaucoup plus léger. On n'exporte que l'encodeur et la tête (texte → 28 logits). La couche factorielle, elle, est appliquée ensuite en `numpy` avec la table **exacte**.

**Quantification int8.** Les poids du modèle sont stockés en nombres à virgule de 32 bits. On les convertit en entiers de 8 bits : le modèle devient environ 4 fois plus léger et 2 à 3 fois plus rapide sur CPU, au prix d'une petite perte de précision. **On mesure cette perte** : si la fidélité baisse trop, on garde la version 32 bits.

In [ ]:
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

model_cpu = copy.deepcopy(model).to("cpu").float().eval()

onnx_path = OUT_DIR / "eleve.onnx"
int8_path = OUT_DIR / "eleve_int8.onnx"
exemple_enc = tokenizer(test_df["texte"].iloc[:2].tolist(), truncation=True, max_length=MAX_LEN,
                        padding=True, return_tensors="pt")
args = (exemple_enc["input_ids"], exemple_enc["attention_mask"])
options = dict(input_names=["input_ids", "attention_mask"], output_names=["logits"],
               dynamic_axes={"input_ids": {0: "lot", 1: "tokens"}, "attention_mask": {0: "lot", 1: "tokens"},
                             "logits": {0: "lot"}},
               opset_version=17)
try:
    torch.onnx.export(model_cpu, args, str(onnx_path), dynamo=False, **options)
except TypeError:                     # anciennes versions de torch sans l'argument dynamo
    torch.onnx.export(model_cpu, args, str(onnx_path), **options)

quantize_dynamic(str(onnx_path), str(int8_path), weight_type=QuantType.QInt8, per_channel=True)
for p in (onnx_path, int8_path):
    print(f"{p.name:<18} {p.stat().st_size / 1e6:>6.0f} Mo")

### Vérifier que l'export ne change rien, puis mesurer l'effet de l'int8

On fait passer 2 000 verbatims du test dans les deux versions ONNX et on compare à vos Z-scores. La version 32 bits doit donner **les mêmes** résultats que PyTorch ; la version int8, des résultats à peine moins bons. Si la part des scores à ±0,10 baisse de plus de quelques points avec l'int8, gardez la version 32 bits.

In [ ]:
def session_cpu(path, n_threads=N_THREADS_CPU):
    so = ort.SessionOptions()
    so.intra_op_num_threads = n_threads
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])


def predire_onnx(session, textes, batch_size=32):
    sorties = []
    for i in range(0, len(textes), batch_size):
        enc = tokenizer(textes[i:i + batch_size], truncation=True, max_length=MAX_LEN,
                        padding=True, return_tensors="np")
        sorties.append(session.run(None, {"input_ids": enc["input_ids"].astype(np.int64),
                                          "attention_mask": enc["attention_mask"].astype(np.int64)})[0])
    return np.concatenate(sorties)


sous_test = test_df.sample(min(2000, len(test_df)), random_state=SEED)
textes_st = sous_test["texte"].tolist()
F_st_ens = scores_enseignant(sous_test)
versions = {"PyTorch (GPU)": F_test_eleve[test_df.index.get_indexer(sous_test.index)]}
for nom, path in [("ONNX 32 bits", onnx_path), ("ONNX int8", int8_path)]:
    versions[nom] = couche_factorielle(predire_onnx(session_cpu(path, n_threads=os.cpu_count()), textes_st))

comparaison = pd.DataFrame({
    nom: {"r moyen": correlations(Fv, F_st_ens).mean(), "à ±0,10": (np.abs(Fv - F_st_ens) <= 0.10).mean()}
    for nom, Fv in versions.items()}).T
display(comparaison.style.format({"r moyen": "{:.4f}", "à ±0,10": "{:.1%}"}))

### Mesurer la vitesse sur CPU

On mesure le temps pour traiter **un verbatim à la fois**, comme en production quand ils arrivent au fil de l'eau, en limitant le calcul à `N_THREADS_CPU` cœurs. Le temps de la couche factorielle (quelques microsecondes) est négligeable.

**Attention :** le processeur du serveur GPU n'est pas celui de la machine de production. Les temps absolus seront différents, mais le rapport entre les versions reste indicatif. Refaites cette mesure sur la machine de production avec la partie 10.

In [ ]:
textes_mesure = test_df["texte"].sample(200, random_state=SEED).tolist()


def chronometrer(fonction, textes):
    for t in textes[:5]:           # échauffement, non compté
        fonction(t)
    durees = []
    for t in textes:
        t0 = time.perf_counter()
        fonction(t)
        durees.append(time.perf_counter() - t0)
    return np.array(durees) * 1000


torch.set_num_threads(N_THREADS_CPU)


@torch.no_grad()
def un_texte_torch(t):
    enc = tokenizer([t], truncation=True, max_length=MAX_LEN, return_tensors="pt")
    model_cpu(enc["input_ids"], enc["attention_mask"])


vitesses = {"PyTorch CPU 32 bits": chronometrer(un_texte_torch, textes_mesure)}
for nom, path in [("ONNX 32 bits", onnx_path), ("ONNX int8", int8_path)]:
    s = session_cpu(path)
    vitesses[nom] = chronometrer(lambda t: predire_onnx(s, [t]), textes_mesure)

display(pd.DataFrame({nom: {"moyenne (ms)": d.mean(), "médiane (ms)": np.median(d),
                            "95e centile (ms)": np.percentile(d, 95)}
                      for nom, d in vitesses.items()}).T.round(0))

## 10. En production

Le dossier `models/eleve_emotions/` contient tout ce qu'il faut :

| Fichier | Rôle |
|---|---|
| `eleve_int8.onnx` (ou `eleve.onnx`) | le modèle : texte → 28 logits |
| `tokenizer/` | le découpage en tokens |
| `couche_factorielle.npz` | les tables de rangs figées et les poids de Bartlett |
| `config_production.json` | noms des émotions et dimensions, `MAX_LEN` |
| `meilleur.pt` | poids PyTorch, pour ré-entraîner ou ré-exporter plus tard |

Sur la machine de production, il suffit d'installer `onnxruntime`, `transformers`, `numpy` et `pandas` (pas besoin de `torch`), de copier ce dossier, et d'utiliser la fonction `predire_verbatims` ci-dessous.

**Deux règles à documenter pour la DITP :**
- **Les tables de rangs ne se recalculent jamais.** Les scores situent chaque verbatim par rapport au corpus de la thèse. Recalculer les rangs avec de nouveaux verbatims changerait rétroactivement tous les scores.
- **Gardez les 28 probabilités en base**, pas seulement les 5 scores : elles permettent d'expliquer chaque résultat, et de vérifier plus tard que la structure factorielle tient toujours.

**Surveillance de la dérive.** Tous les trimestres, faites annoter quelques centaines de nouveaux verbatims par le zero-shot (sur GPU) et comparez-les à l'élève avec les mesures de la partie 8. Si la fidélité baisse, ré-entraînez l'élève en ajoutant ces nouveaux exemples.

In [ ]:
tokenizer.save_pretrained(OUT_DIR / "tokenizer")
config_prod = {"student_model": STUDENT_MODEL, "max_len": MAX_LEN,
               "emotions": EMOTIONS, "dimensions": DIMENSIONS, "facteurs": FACTEURS,
               "resultats_test": res_dim.round(4).to_dict()}
with open(OUT_DIR / "config_production.json", "w", encoding="utf-8") as f:
    json.dump(config_prod, f, ensure_ascii=False, indent=1)
print("Fichiers prêts dans", OUT_DIR.resolve())

### La fonction de production

Cette cellule est **autonome** : elle ne dépend de rien d'autre dans le notebook. Vous pouvez la copier telle quelle dans un script sur la machine de production. Elle renvoie un tableau avec les 28 probabilités et les 5 scores factoriels pour chaque verbatim.

In [ ]:
import json
from pathlib import Path

import numpy as np
import onnxruntime as ort
import pandas as pd
from transformers import AutoTokenizer

DOSSIER_MODELE = Path("models/eleve_emotions")
N_THREADS = 4

with open(DOSSIER_MODELE / "config_production.json", encoding="utf-8") as f:
    CONFIG = json.load(f)
TOKENIZER = AutoTokenizer.from_pretrained(DOSSIER_MODELE / "tokenizer")
COUCHE = np.load(DOSSIER_MODELE / "couche_factorielle.npz")
TABLES = [(COUCHE[f"x_{j}"], COUCHE[f"y_{j}"]) for j in range(len(CONFIG["emotions"]))]
_so = ort.SessionOptions()
_so.intra_op_num_threads = N_THREADS
SESSION = ort.InferenceSession(str(DOSSIER_MODELE / "eleve_int8.onnx"), _so,
                               providers=["CPUExecutionProvider"])


def predire_verbatims(textes):
    """textes : liste de chaînes (Titre + "\n\n" + Description). Renvoie un DataFrame 28 + 5 colonnes."""
    enc = TOKENIZER(list(textes), truncation=True, max_length=CONFIG["max_len"],
                    padding=True, return_tensors="np")
    logits = SESSION.run(None, {"input_ids": enc["input_ids"].astype(np.int64),
                                "attention_mask": enc["attention_mask"].astype(np.int64)})[0].astype(np.float64)
    rangs = np.column_stack([np.interp(logits[:, j], x, y) for j, (x, y) in enumerate(TABLES)])
    scores = rangs @ COUCHE["W_final"] + COUCHE["c_final"]
    return pd.concat([pd.DataFrame(1 / (1 + np.exp(-logits)), columns=CONFIG["emotions"]),
                      pd.DataFrame(scores, columns=CONFIG["dimensions"])], axis=1)


resultat = predire_verbatims([
    "Démarche simple\n\nTout s'est très bien passé, dossier traité en deux jours.",
    "Inadmissible\n\nTrois mois sans réponse, personne au téléphone, je ne sais plus quoi faire.",
])
resultat[CONFIG["dimensions"]].round(2)